# 02 — Bias Analysis (Gender Disparate Impact)

This notebook evaluates potential gender bias in loan approval decisions using the Disparate Impact (DI) ratio.

The analysis follows the 80% rule, a commonly used screening test in fairness assessments.

Important note:
Disparate Impact measures statistical disparity, not causality or intent. It is an initial diagnostic indicator that may require further investigation.

## STEP 0 — Setup and Load Clean Dataset

In [0]:


from pyspark.sql import functions as F

# Load cleaned dataset produced in the Data Quality phase
df = spark.table("applications_gold")

print("Total rows:", df.count())
display(df.limit(5))

Total rows: 500


_id,gender,date_of_birth,zip_code,annual_income,credit_history_months,debt_to_income,savings_balance,loan_approved,interest_rate,approved_amount,rejection_reason,gender_standardized,dob_parsed,email_valid,ssn_hash,ip_hash,email_hash,name_hash
app_307,M,1990/07/26,10097,53000.0,83,0.1,17420.0,true,6.1,57000.0,null,Male,1990-07-26,true,23dfc1e1a65a17a682387be7c1226edfc89ec6ba5fe875fd01c306a2e72c7889,42802298be84e944d13520b5e34a674fbfd538eeaa5702a9f7727ab2204adbc3,27b7e80c92c43adb99d98be1e98287a380aa3122359e101eb498bf058cd6f0b4,0941a6d391bdf165f75bba91ef0d508614e991d3e70f0b196d225fb579de7f80
app_133,Male,1989-10-07,10022,70000.0,72,0.16,33883.0,false,null,null,algorithm_risk_score,Male,1989-10-07,true,c5a413e66ffacd2e2fa3a41576dbd7b9002492b16c39d1ef8c14a0021a58b2d0,c4cfe51952860b89f4b59e081520bef4a0544452e475dd95942bbdc4fe8f5734,05cbb9ed24c253551b4d2862fc818901e8482870d1fc6679c84ea22fb5204465,288359072dca7649fba00349fa2669bae0b0cda3576be05af6e6f136374cb8ca
app_130,Female,03/10/1981,90252,135000.0,62,0.09,30902.0,false,null,null,algorithm_risk_score,Female,1981-10-03,true,2287417cb87f4e1a3a83118f9304f13249739aedf3559927b4789584e2e67e5d,e22fbceba00a891dd00be2507d8ee05c1e33ad95ed6ffc586f87ed39caa1833f,d67b96cd8d564d8e6b8495e281caad54509c163482d067319ed2f449b837f02b,da43d358931f763c1db72a0c9df44f32838ed7e91d4700158642ab58c421d931
app_404,Male,1985-11-10,10017,91000.0,31,0.32,2729.0,true,5.9,69000.0,null,Male,1985-11-10,true,bf6fae78f965cbe97b3583f371172603eae6954ecbde74b1de43b8a1167d700a,0f502226dbcb2f476b29ab4624069ab407ef30d449b1776058ef9c73926c013f,da4e58b7eb9fee84c83462f815f79110ecefa054f30745812819098b7198eda6,ea0c58a3d4aa7975415754c828cfcfc045742240b28f36793fdffc0986d69a9c
app_407,Male,1990/11/09,10076,74000.0,31,0.26,32341.0,false,null,null,algorithm_risk_score,Male,1990-11-09,true,46cc64e01b06dccf042e609bf553a56226dd28ccee63a26dcd5d680f2efbcd67,519a8badc47f5548d30ba2e63f3913614f88c5a7af54df5a3ec85db00b8a1a63,a2175a08cffdeb69b55778388e73e9f2cf964112618622ab2a5e31294a92f474,aa43bc2fd3f024d1c3a46cb066fe3a73e29920bd7cf547840dd7fbef9e24e041


This step loads the cleaned dataset generated during the Data Quality phase.
Bias analysis should be performed on remediated data to ensure that fairness metrics are not distorted by data quality issues.


## STEP 1A — Approval Rate by Gender


In [0]:


approval_by_gender = (
    df.groupBy("gender_standardized")
      .agg(
          F.count("*").alias("total_applications"),
          F.sum(F.col("loan_approved").cast("int")).alias("approved_count")
      )
      .withColumn(
          "approval_rate",
          F.col("approved_count") / F.col("total_applications")
      )
)

display(approval_by_gender)

gender_standardized,total_applications,approved_count,approval_rate
Male,247,163,0.659919028340081
Female,251,127,0.5059760956175299
Unknown,2,2,1.0


## Approval Rate Calculation

Approval rate is calculated as:

approval_rate = approved_count / total_applications

This metric represents the probability of approval per gender group and is required for Disparate Impact computation.

## STEP 1B — Sample Size Validation

In [0]:

approval_by_gender.select(
    "gender_standardized",
    "total_applications"
).display()

gender_standardized,total_applications
Male,247
Female,251
Unknown,2



Before computing Disparate Impact, we verify that each group has sufficient observations.

Very small sample sizes may distort fairness metrics and produce unstable DI ratios.


## STEP 2 — Disparate Impact (DI) Calculation


In [0]:
rates = approval_by_gender.collect()

rate_dict = {
    row["gender_standardized"]: row["approval_rate"]
    for row in rates
}

male_rate = rate_dict.get("Male")
female_rate = rate_dict.get("Female")

di_ratio = female_rate / male_rate if male_rate else None

print("Male approval rate:", male_rate)
print("Female approval rate:", female_rate)
print("Disparate Impact Ratio (Female / Male):", di_ratio)

Male approval rate: 0.659919028340081
Female approval rate: 0.5059760956175299
Disparate Impact Ratio (Female / Male): 0.7667245129909809


## Disparate Impact Formula and 80% Rule

Disparate Impact is calculated as:

DI = Approval Rate (Female) / Approval Rate (Male)

According to the 80% rule:

- If DI ≥ 0.80 → No statistical evidence of adverse impact  
- If DI < 0.80 → Potential adverse impact exists  

The 80% threshold is commonly used for fairness screening in practice.

## STEP 2B — Automated Interpretation (80% Rule)

In [0]:

if di_ratio is None:
    print("DI ratio could not be computed due to missing reference group rate.")
elif di_ratio < 0.8:
    print("⚠ Potential adverse impact detected under the 80% rule (DI < 0.80).")
elif di_ratio <= 1.25:
    print("✓ No strong evidence of adverse impact under the 80% rule (0.80 ≤ DI ≤ 1.25).")
else:
    print("⚠ DI > 1.25, possible reverse disparity signal. Further investigation recommended.")

⚠ Potential adverse impact detected under the 80% rule (DI < 0.80).



# STEP 3 — Statistical Interpretation Note


Disparate Impact identifies proportional differences between groups, not causality.

Observed disparity may result from:
- Legitimate financial risk variables
- Data imbalance
- Feature interactions
- Historical structural patterns

A regulatory-grade fairness audit would require multivariate modeling, controlled testing, or counterfactual analysis.


# STEP 4 — Final Bias Assessment Conclusion


This analysis computed approval rates by gender and derived the Disparate Impact ratio.

If DI ≥ 0.80, the system does not exhibit strong statistical evidence of gender-based adverse impact under the 80% rule.
If DI < 0.80, the result signals potential adverse impact and should be investigated further.

This completes the required basic bias detection task for the project scope.